In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from causalnex.structure.notears import from_pandas

In [2]:
df_ee = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/ee_2007_2015_bn.csv")

In [3]:
df_ee = df_ee.drop(["Unnamed: 0"], axis = 1)

In [4]:
df_ee

,FDI,QOR,QOA,EIAP,PDPS,PGP,VABI,GCFP,GGFC,GDPCG,...,FEM,LFG,EPFRG,GGAG,PMSI,JI,DWCP,PR,DBPG,EUPC
0,13.51,3.78,4.99,4.78,32.0,-0.46,26.84,40.005894,15.899159,8.062766,...,73.842857,0.00,8.80,3732.0,56.666667,0.955,86.112980,90,0.0,5.509541e+07
1,8.12,4.00,5.10,3.90,32.0,-0.27,26.13,31.633406,18.436467,-4.870579,...,73.842857,0.01,44.91,-2269.0,56.666667,0.955,86.252845,90,0.0,8.479186e+07
2,9.51,4.22,4.87,4.05,31.0,-0.19,23.28,21.008775,20.533284,-14.465498,...,69.900000,-0.01,340.12,-3519.0,56.666667,0.955,85.403987,90,0.0,3.568504e+08
3,13.28,4.53,4.55,4.22,31.0,-0.23,24.03,21.587678,19.561652,2.683662,...,67.400000,-0.01,507.33,4528.0,56.666667,0.955,85.415456,80,0.0,3.969299e+08
4,4.80,4.49,4.39,4.41,31.0,-0.30,25.35,25.468166,18.294990,7.934147,...,68.600000,0.01,132.91,-34.0,56.666667,0.955,85.358224,80,0.0,1.797442e+08
5,7.69,4.22,4.46,4.49,31.0,-0.36,24.63,29.080507,18.024258,4.043772,...,71.100000,-0.01,285.06,-1474.0,56.666667,0.955,85.374812,80,0.0,4.990624e+08
6,4.31,4.20,4.14,4.26,30.0,-0.36,24.68,26.869712,18.399173,2.119282,...,71.600000,0.00,-239.99,1866.0,56.666667,0.955,85.428313,85,0.0,5.140841e+08
7,6.59,4.39,3.78,3.86,30.0,-0.26,24.11,26.727616,18.439013,3.594116,...,71.900000,0.00,169.83,-1231.0,56.666667,0.955,85.462140,90,0.0,2.871504e+08
8,-3.07,4.53,3.73,4.03,30.0,0.07,23.12,24.640961,19.223265,1.771417,...,73.700000,0.00,173.78,-3206.0,56.666667,0.955,85.640301,90,0.0,6.873445e+07


In [5]:
discretised_ee = pd.DataFrame(index=df_ee.index)

for col in df_ee.columns:
    no_unique = df_ee[col].nunique()
    
    if no_unique <= 1:
        discretised_ee[col] = 0
    else:
        try:
            discretised_ee[col] = pd.qcut(
                df_ee[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_ee[col] = df_ee[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_ee)

Discretised data:
   FDI  QOR  QOA  EIAP  PDPS  PGP  VABI  GCFP  GGFC  GDPCG  ...  FEM  LFG  \
0    2    0    2     2     2    0     2     2     0      2  ...    2    1   
1    1    0    2     0     2    1     2     2     1      0  ...    2    2   
2    2    1    2     1     1    2     0     0     2      0  ...    0    0   
3    2    2    1     1     1    2     0     0     2      1  ...    0    0   
4    0    2    1     2     1    1     2     1     0      2  ...    0    2   
5    1    1    1     2     1    0     1     2     0      2  ...    1    0   
6    0    0    0     1     0    0     1     1     1      1  ...    1    1   
7    1    1    0     0     0    1     1     1     1      1  ...    1    1   
8    0    2    0     0     0    2     0     0     2      0  ...    2    1   

   EPFRG  GGAG  PMSI  JI  DWCP  PR  DBPG  EUPC  
0      0     2     0   0     2   1     0     0  
1      0     0     0   0     2   1     0     0  
2      2     0     0   0     0   1     0     1  
3      2     2 

In [6]:
sm = from_pandas(discretised_ee)

In [7]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/01_fully_connected_ee.html")

Graphs/01_fully_connected_ee.html


In [8]:
sm.remove_edges_below_threshold(0.8)
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)
viz.show("Graphs/01_thresholded_ee.html")

Graphs/01_thresholded_ee.html


In [9]:
# sm = sm.get_largest_subgraph()
# viz = plot_structure(
#     sm,
#     all_node_attributes=NODE_STYLE.WEAK,
#     all_edge_attributes=EDGE_STYLE.WEAK,
# )
# viz.show("Graphs/01_largest_subgraph_ee_1.html")

In [11]:
edge_data = []
for u, v, data in sm.edges(data=True):
    edge_data.append({
        'From': u, 
        'To': v, 
        'Weight': data.get('weight', 'N/A')
    })

df_edges = pd.DataFrame(edge_data).sort_values(by='Weight', ascending=False)
print(df_edges)
df_edges.to_csv("ee.csv")

    From     To     Weight
4   VABI   EIAP  13.173259
7   VABI  GDPCG   4.788225
34  GGFC   EIAP   4.196609
46   FEM   EIAP   3.826970
64  DWCP   GGAG   1.999657
..   ...    ...        ...
55   LFG  GDPCG  -2.997504
61  DWCP   EIAP  -3.411702
67    PR   EIAP  -3.808942
53   LFG   EIAP  -6.778609
19  GCFP   EIAP  -7.850679

[72 rows x 3 columns]
